# IFLS5 Weighted Analysis
Survey-weighted logistic regression using `pwt14xa` (IFLS5 2014).

In [ ]:
import pandas as pd
import numpy as np
import patsy
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)

## 1. Load Raw Data

In [ ]:
def load_stata(path, cols):
    df = pd.read_stata(path, columns=cols)
    for col in df.columns:
        if isinstance(df[col].dtype, pd.CategoricalDtype):
            df[col] = df[col].astype(object)
    return df

cov    = load_stata('adult_a/b3a_cov.dta',       ['pidlink','hhid14','marstat','age','sex'])
dl1    = load_stata('adult_a/b3a_dl1.dta',       ['pidlink','dl01f','dl06'])
tk2    = load_stata('adult_a/b3a_tk2.dta',       ['pidlink','tk25a1','tk25a2'])
ptrack = load_stata('tracking_book/ptrack.dta',  ['pidlink','pwt14xa'])
bk_sc1 = load_stata('control_book/bk_sc1.dta',  ['hhid14','sc05'])   # urban/rural

print('cov   :', cov.shape)
print('dl1   :', dl1.shape)
print('tk2   :', tk2.shape)
print('ptrack:', ptrack.shape)
print('bk_sc1:', bk_sc1.shape)
print('Urban/Rural counts:', bk_sc1['sc05'].value_counts().to_dict())

## 2. Merge Datasets

In [ ]:
df = cov.merge(dl1,    on='pidlink', how='inner') \
        .merge(tk2,    on='pidlink', how='inner') \
        .merge(ptrack, on='pidlink', how='left') \
        .merge(bk_sc1, on='hhid14',  how='left')   # urban/rural via household ID

df = df.rename(columns={
    'tk25a1': 'monthly_income',
    'tk25a2': 'yearly_income',
    'dl01f' : 'ethnicity',
    'dl06'  : 'edu_level',
    'sc05'  : 'urban_rural',
})
print('Merged:', df.shape)
print('urban_rural non-null:', df['urban_rural'].notna().sum())
print('urban_rural values:', df['urban_rural'].value_counts().to_dict())

## 3. Clean Variables

In [ ]:
# â”€â”€ Age â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
df = df[df['age'].astype(str) != "998:Don't Know"].copy()
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df = df[df['age'] >= 18]

# â”€â”€ Marital status â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
marstat_map = {
    '1:Not yet married': 'Never Married',
    '2:Married'        : 'Married',
    '3:Separated'      : 'Separated',
    '4:Divorced'       : 'Divorced',
    '5:Widowed'        : 'Widowed',
    '6:Cohabitate'     : 'Cohabitate',
}
df['marstat'] = df['marstat'].astype(str).map(marstat_map)
df = df[df['marstat'].notna()]

# â”€â”€ Sex â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
df['sex'] = df['sex'].astype(str).map({'1:Male': 'Male', '3:Female': 'Female'})
df = df[df['sex'].isin(['Male', 'Female'])]

# â”€â”€ Ethnicity â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
ethnic_map = {
    'A':'Jawa','B':'Sunda','C':'Bali','D':'Batak','E':'Bugis','F':'Tionghoa',
    'G':'Madura','H':'Sasak','I':'Minang','J':'Banjar','K':'Bima-Dompu',
    'L':'Makassar','M':'Nias','N':'Palembang','O':'Sumbawa','P':'Toraja',
    'Q':'Betawi','R':'Dayak','S':'Melayu','T':'Komering','U':'Ambon',
    'A1':'Manado','B1':'Aceh','C1':'Other South Sumatera','D1':'Banten',
    'E1':'Cirebon','F1':'Gorontalo','G1':'Kutai','V':'Other',
}
df['ethnicity'] = df['ethnicity'].astype(str).map(ethnic_map).fillna('Other')

# â”€â”€ Education: map IFLS level labels â†’ years of schooling â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Years at START of each level (base years before entering that level)
level_base_years = {
    '1:Never school'                                       : 0,
    '2:Elementary school'                                  : 0,
    '3:Junior high general'                                : 6,
    '4:Junior high vocational'                             : 6,
    '5:Senior high general'                                : 9,
    '6:Senior high vocational'                             : 9,
    '11:Adult education A'                                 : 0,
    '12:Adult education B'                                 : 6,
    '13:Open university'                                   : 12,
    '14:Islamic School (pesantren)'                        : 9,
    '15:Adult education C'                                 : 9,
    '60:College (D1,D2,D3)'                               : 12,
    '61:University S1'                                     : 12,
    '62:University S2'                                     : 16,
    '63:University S3'                                     : 18,
    '72:Islamic Elementary School (Madrasah Ibtidaiyah)'  : 0,
    '73:Islamic Junior/High School (Madrasah Tsanawiyah)' : 6,
    '74:Islamic Senior/High School (Madrasah Tsanawiyah)' : 9,
}
# Full years upon graduation
level_full_years = {
    '1:Never school'                                       : 0,
    '2:Elementary school'                                  : 6,
    '3:Junior high general'                                : 9,
    '4:Junior high vocational'                             : 9,
    '5:Senior high general'                                : 12,
    '6:Senior high vocational'                             : 12,
    '11:Adult education A'                                 : 6,
    '12:Adult education B'                                 : 9,
    '13:Open university'                                   : 15,
    '14:Islamic School (pesantren)'                        : 12,
    '15:Adult education C'                                 : 12,
    '60:College (D1,D2,D3)'                               : 15,
    '61:University S1'                                     : 16,
    '62:University S2'                                     : 18,
    '63:University S3'                                     : 21,
    '72:Islamic Elementary School (Madrasah Ibtidaiyah)'  : 6,
    '73:Islamic Junior/High School (Madrasah Tsanawiyah)' : 9,
    '74:Islamic Senior/High School (Madrasah Tsanawiyah)' : 12,
}

def compute_education(row):
    level = str(row['edu_level'])
    grade = str(row['dl07'])
    base = level_base_years.get(level, np.nan)
    full = level_full_years.get(level, np.nan)
    if pd.isna(base):
        return np.nan
    if grade == '7:Graduated':
        return full
    elif grade in ["0:Did not complete first grade at that level"]:
        return base
    elif grade in ["98:Don't Know", "99:MISSING", "nan"]:
        return full  # fallback to full years if unknown
    else:
        try:
            g = float(grade)
            return base + g
        except:
            return full

def compute_education(row):
    level = str(row['edu_level'])
    if 'dl07' in row:
        grade = str(row['dl07'])
    else:
        grade = '7:Graduated'  # assume graduated if grade column not available
    base = level_base_years.get(level, np.nan)
    full = level_full_years.get(level, np.nan)
    if pd.isna(base):
        return np.nan
    if grade == '7:Graduated':
        return full
    elif grade in ["0:Did not complete first grade at that level"]:
        return base
    elif grade in ["98:Don't Know", "99:MISSING", "nan"]:
        return full  # fallback to full years if unknown
    else:
        try:
            g = float(grade)
            return base + g
        except:
            return full

df['education'] = df.apply(compute_education, axis=1)

# â”€â”€ Urban/Rural: 1 = Urban, 0 = Rural â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
df['urban'] = df['urban_rural'].astype(str).map({'1:Urban': 1, '2:Rural': 0})

print('Shape after cleaning:', df.shape)
print('education non-null:', df['education'].notna().sum())
print('urban non-null:', df['urban'].notna().sum())
print('urban values:', df['urban'].value_counts().to_dict())

## 4. Income Transform & Derived Variables

In [ ]:
IFLS_MISSING = [999999997, 999999998, 999999999]
df['monthly_income'] = pd.to_numeric(df['monthly_income'], errors='coerce').replace(IFLS_MISSING, np.nan)
df['yearly_income']  = pd.to_numeric(df['yearly_income'],  errors='coerce').replace(IFLS_MISSING, np.nan)
df['log_income']     = np.log1p(df['monthly_income'])
df['age_squared']    = df['age'] ** 2

print(df[['monthly_income','log_income','age','education','urban']].describe())

## 4b. Income Distribution â€” Before vs. After Cleaning (by Sex)

In [ ]:
IFLS_MISSING = [999999997, 999999998, 999999999]

# â”€â”€ Load raw income BEFORE any cleaning â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
tk2_raw = load_stata("adult_a/b3a_tk2.dta", ["pidlink","tk25a1"])
cov_raw = load_stata("adult_a/b3a_cov.dta", ["pidlink","sex","age"])
cov_raw["sex"] = cov_raw["sex"].astype(str).map({"1:Male":"Male","3:Female":"Female"})
cov_raw["age"] = pd.to_numeric(cov_raw["age"], errors="coerce")
raw = cov_raw[cov_raw["sex"].isin(["Male","Female"]) & (cov_raw["age"]>=18)].merge(tk2_raw, on="pidlink", how="inner")
raw["income_raw"] = pd.to_numeric(raw["tk25a1"], errors="coerce")

print("=" * 60)
print("BEFORE CLEANING (raw tk25a1)")
print("=" * 60)
for sex in ["Female", "Male"]:
    sub = raw[raw["sex"]==sex]["income_raw"]
    nan_true  = sub.isna().sum()
    nan_ifls  = sub.isin(IFLS_MISSING).sum()
    zero      = (sub==0).sum()
    valid     = sub[~sub.isin(IFLS_MISSING) & sub.notna() & (sub>0)]
    print(f"--- {sex} (N={len(sub):,}) ---")
    print(f"  True NaN (did not answer) : {nan_true:,} ({nan_true/len(sub)*100:.1f}%)")
    print(f"  IFLS missing codes (999..) : {nan_ifls:,} ({nan_ifls/len(sub)*100:.1f}%)")
    print(f"  Zero income                : {zero:,} ({zero/len(sub)*100:.1f}%)")
    print(f"  Valid (>0, non-missing)    : {len(valid):,} ({len(valid)/len(sub)*100:.1f}%)")
    print(f"  Max (raw)                  : {sub.max():,.0f}")

print()
print("=" * 60)
print("AFTER CLEANING (IFLS codes removed, zero/NaN dropped)")
print("=" * 60)
raw["income_clean"] = raw["income_raw"].replace(IFLS_MISSING, np.nan)
valid_df = raw.dropna(subset=["income_clean"])
valid_df = valid_df[valid_df["income_clean"] > 0]
for sex in ["Female", "Male"]:
    sub = valid_df[valid_df["sex"]==sex]["income_clean"]
    total = len(raw[raw["sex"]==sex])
    print(f"--- {sex}: {len(sub):,} of {total:,} kept ({len(sub)/total*100:.1f}%) | Dropped: {total-len(sub):,} ({(total-len(sub))/total*100:.1f}%) ---")
    print(sub.describe().apply(lambda x: f"{x:,.0f}").to_string())
    print(f"  Log income mean : {np.log1p(sub).mean():.3f}")
    print(f"  Log income std  : {np.log1p(sub).std():.3f}")

# â”€â”€ Bar chart: kept vs dropped by sex â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, sex, color in zip(axes, ["Female","Male"], ["#e74c3c","#2980b9"]):
    total = len(raw[raw["sex"]==sex])
    kept  = len(valid_df[valid_df["sex"]==sex])
    ax.bar(["Dropped","Kept"], [total-kept, kept],
           color=["#bdc3c7", color], edgecolor="white")
    ax.set_title(f"{sex}: Income Coverage", fontsize=12)
    ax.set_ylabel("N respondents")
    for i, v in enumerate([total-kept, kept]):
        ax.text(i, v+50, f"{v:,}\n({v/total*100:.1f}%)", ha="center", fontsize=9)
plt.suptitle("Selection Bias: Who Is Dropped When Income Is Required", fontsize=13)
plt.tight_layout()
plt.savefig("income_selection_bias.png", dpi=150)
plt.show()

# â”€â”€ Log income distribution after cleaning â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fig, ax = plt.subplots(figsize=(9, 4))
for sex, color in [("Female","#e74c3c"),("Male","#2980b9")]:
    sub = np.log1p(valid_df[valid_df["sex"]==sex]["income_clean"])
    ax.hist(sub, bins=40, alpha=0.55, color=color, label=sex, edgecolor="none")
ax.set_xlabel("Log Monthly Income", fontsize=12)
ax.set_ylabel("Frequency", fontsize=12)
ax.set_title("Log Income Distribution After Cleaning â€” by Sex", fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig("log_income_distribution.png", dpi=150)
plt.show()

## 5. Outcome & Dummy Variables

In [ ]:
df['ever_married'] = (df['marstat'] != 'Never Married').astype(int)
df['is_divorced']  = df['marstat'].isin(['Divorced','Separated']).astype(int)
df['is_jawa']      = (df['ethnicity'] == 'Jawa').astype(int)
df['is_sunda']     = (df['ethnicity'] == 'Sunda').astype(int)

print('ever_married:', df['ever_married'].value_counts().to_dict())
print('is_divorced :', df['is_divorced'].value_counts().to_dict())
print('urban rate  :', round(df['urban'].mean(), 3))

## 6. Final Sample â€” Drop Missing & Save

In [ ]:
keep = ['pidlink','sex','age','age_squared','education','urban',
        'monthly_income','log_income','marstat',
        'ever_married','is_divorced','is_jawa','is_sunda','pwt14xa']

print('NaN per column before dropna:')
print(df[keep].isna().sum())

df_clean = df[keep].dropna().reset_index(drop=True)

print('\nFinal sample :', df_clean.shape)
print('Male         :', (df_clean['sex']=='Male').sum())
print('Female       :', (df_clean['sex']=='Female').sum())
print('Urban        :', df_clean['urban'].mean().round(3))

df_clean.to_csv('clean_data/data_clean_weighted.csv', index=False)

## 7. Weighted Descriptive Statistics (Table 1)

In [ ]:
def weighted_mean(series, weights):
    mask = series.notna() & weights.notna()
    return np.average(series[mask], weights=weights[mask])

def weighted_se(series, weights):
    mask = series.notna() & weights.notna()
    w, x = weights[mask].values, series[mask].values
    wm = np.average(x, weights=w)
    return np.sqrt(np.average((x - wm)**2, weights=w) / mask.sum())

vars_desc = {
    'Age'                  : 'age',
    'Monthly Income (IDR)' : 'monthly_income',
    'Education (years)'    : 'education',
    'Urban (prop)'         : 'urban',
    'Ever Married (prop)'  : 'ever_married',
    'Ever Divorced (prop)' : 'is_divorced',
    'Javanese (prop)'      : 'is_jawa',
    'Sundanese (prop)'     : 'is_sunda',
}

rows = []
for label, col in vars_desc.items():
    for grp, gdf in [('Women', df_clean[df_clean['sex']=='Female']),
                     ('Men',   df_clean[df_clean['sex']=='Male'])]:
        mask = gdf[col].notna() & gdf['pwt14xa'].notna()
        n_obs = mask.sum()
        rows.append({'Variable': label, 'Group': grp,
                     'N'   : n_obs,
                     'Mean': round(weighted_mean(gdf[col], gdf['pwt14xa']), 3),
                     'SE'  : round(weighted_se(gdf[col],   gdf['pwt14xa']), 3)})

tbl1 = pd.DataFrame(rows).pivot(index='Variable', columns='Group', values=['N','Mean','SE'])
tbl1.columns = [f'{s} {g}' for s, g in tbl1.columns]
tbl1 = tbl1[['N Women','Mean Women','SE Women','N Men','Mean Men','SE Men']]
tbl1['N Women'] = tbl1['N Women'].astype(int)
tbl1['N Men']   = tbl1['N Men'].astype(int)
print('\n=== Table 1: Weighted Means and Standard Errors ===')
print(tbl1.to_string())


In [ ]:
def w_mean(s, w):
    mask = s.notna() & w.notna()
    return np.average(s[mask], weights=w[mask])

def w_std(s, w):
    mask = s.notna() & w.notna()
    x, wt = s[mask].values, w[mask].values
    m = np.average(x, weights=wt)
    return np.sqrt(np.average((x - m)**2, weights=wt))

vars_info = {
    'Age (years)'           : ('age',           'continuous'),
    'Monthly Income (IDR)'  : ('monthly_income', 'continuous'),
    'Log Monthly Income'    : ('log_income',     'continuous'),
    'Education (years)'     : ('education',      'continuous'),
    'Ever Married (=1)'     : ('ever_married',   'binary'),
    'Ever Divorced (=1)'    : ('is_divorced',    'binary'),
    'Javanese (=1)'         : ('is_jawa',        'binary'),
    'Sundanese (=1)'        : ('is_sunda',       'binary'),
    'Urban (=1)'            : ('urban',          'binary'),
}

rows = []
for label, (col, vtype) in vars_info.items():
    row = {'Variable': label}
    for grp, gname in [('Female', 'Women'), ('Male', 'Men'), ('all', 'Total')]:
        sub = df_clean if grp == 'all' else df_clean[df_clean['sex'] == grp]
        mask = sub[col].notna() & sub['pwt14xa'].notna()
        n_obs = mask.sum()
        m = w_mean(sub[col], sub['pwt14xa'])
        s = w_std(sub[col],  sub['pwt14xa'])
        row[f'{gname} N']   = n_obs
        row[f'{gname} Mean'] = round(m, 3) if vtype == 'binary' else round(m, 2)
        row[f'{gname} SD']   = round(s, 3) if vtype == 'binary' else round(s, 2)
    rows.append(row)

tbl_desc = pd.DataFrame(rows).set_index('Variable')
tbl_desc = tbl_desc[['Women N','Women Mean','Women SD','Men N','Men Mean','Men SD','Total N','Total Mean','Total SD']]

n_f = (df_clean['sex']=='Female').sum()
n_m = (df_clean['sex']=='Male').sum()
n_t = len(df_clean)

print('=== TABLE 1b: WEIGHTED DESCRIPTIVE STATISTICS — ALL MODEL VARIABLES ===\n')
hdr1 = f"{'Variable':<26} {'Women':>33} {'Men':>33} {'Total':>33}"
hdr2 = f"{'':26} {'N':>11} {'Mean':>11} {'(SD)':>11} {'N':>11} {'Mean':>11} {'(SD)':>11} {'N':>11} {'Mean':>11} {'(SD)':>11}"
print(hdr1)
print(hdr2)
print('-' * 125)
for _, r in tbl_desc.reset_index().iterrows():
    wn = f"{int(r['Women N']):,}"
    wm = f"{r['Women Mean']:,.2f}" if r['Women Mean'] > 1 else f"{r['Women Mean']:.3f}"
    ws = f"({r['Women SD']:,.2f})" if r['Women SD'] > 1 else f"({r['Women SD']:.3f})"
    mn = f"{int(r['Men N']):,}"
    mm = f"{r['Men Mean']:,.2f}" if r['Men Mean'] > 1 else f"{r['Men Mean']:.3f}"
    ms = f"({r['Men SD']:,.2f})" if r['Men SD'] > 1 else f"({r['Men SD']:.3f})"
    tn = f"{int(r['Total N']):,}"
    tm = f"{r['Total Mean']:,.2f}" if r['Total Mean'] > 1 else f"{r['Total Mean']:.3f}"
    ts = f"({r['Total SD']:,.2f})" if r['Total SD'] > 1 else f"({r['Total SD']:.3f})"
    print(f"{r['Variable']:<26} {wn:>11} {wm:>11} {ws:>11} {mn:>11} {mm:>11} {ms:>11} {tn:>11} {tm:>11} {ts:>11}")
print('-' * 125)
print(f"{'Total N':<26} {n_f:>11,} {'':22} {n_m:>11,} {'':22} {n_t:>11,}")
print('\nNote: Weighted means and SDs using IFLS5 probability weights (pwt14xa).')
print('Binary variables reported as proportions; N shows non-missing observations.')


## 7b. Descriptive Statistics â€” All Model Variables (Weighted Mean & SD)

## 8. Weighted Logistic Regression â€” Helper Functions

In [ ]:
def run_weighted_model(df_sub, outcome, formula_rhs, label):
    pred_cols = [c.strip() for c in formula_rhs.replace('~','').split('+')]
    df_m = df_sub.dropna(subset=[outcome] + pred_cols + ['pwt14xa']).copy()
    n = len(df_m)
    w = df_m['pwt14xa'].values
    w_norm = w / w.sum() * n
    y, X = patsy.dmatrices(f'{outcome} {formula_rhs}', data=df_m, return_type='dataframe')
    model = sm.Logit(y.values.ravel(), X, freq_weights=w_norm).fit(disp=False)
    pr2 = 1 - model.llf / model.llnull
    res = pd.DataFrame({'Coef.': model.params, 'Std.Err.': model.bse,
                        'z': model.tvalues, 'P>|z|': model.pvalues}, index=X.columns)
    print(f'\n--- {label} (N={n:,}) ---')
    print(res.round(3).to_string())
    print(f'Pseudo RÂ²: {pr2:.4f}')
    return model, X.columns.tolist()


def format_results_table(m_f, cols_f, m_m, cols_m, title):
    results = {}
    for label, model, cols in [('Women', m_f, cols_f), ('Men', m_m, cols_m)]:
        results[label] = {}
        for i, col in enumerate(cols):
            stars = '***' if model.pvalues[i]<0.001 else '**' if model.pvalues[i]<0.01 else '*' if model.pvalues[i]<0.05 else ''
            results[label][col] = f'{model.params[i]:.3f}{stars}\n({model.bse[i]:.3f})'
    tbl = pd.DataFrame(results)
    tbl.loc['Pseudo RÂ²'] = [f'{1-m_f.llf/m_f.llnull:.3f}', f'{1-m_m.llf/m_m.llnull:.3f}']
    tbl.loc['N']         = [f'{int(m_f.nobs):,}', f'{int(m_m.nobs):,}']
    print(f'\n=== {title} ===')
    print(tbl.to_string())
    print('\n* p<0.05  ** p<0.01  *** p<0.001')
    return tbl

## 9. Split by Sex

In [ ]:
df_male   = df_clean[df_clean['sex']=='Male'].copy()
df_female = df_clean[df_clean['sex']=='Female'].copy()
print('Male:', len(df_male), '| Female:', len(df_female))

## 10. Model 1 â€” Ever Married (Base: Without Urban/Rural)

In [ ]:
formula_base = '~ log_income + education + age + age_squared + is_jawa + is_sunda'

m1_f, cols1_f = run_weighted_model(df_female, 'ever_married', formula_base, 'Ever Married â€” Women (Base)')
m1_m, cols1_m = run_weighted_model(df_male,   'ever_married', formula_base, 'Ever Married â€” Men (Base)')
tbl_m1 = format_results_table(m1_f, cols1_f, m1_m, cols1_m, 'TABLE 2: EVER MARRIED â€” BASE MODEL')

## 11. Model 2 â€” Ever Divorced (Base: Without Urban/Rural)

In [ ]:
df_em   = df_clean[df_clean['ever_married']==1].copy()
df_em_f = df_em[df_em['sex']=='Female'].copy()
df_em_m = df_em[df_em['sex']=='Male'].copy()
print('Ever married â€” Female:', len(df_em_f), '| Divorced:', df_em_f['is_divorced'].sum())
print('Ever married â€” Male  :', len(df_em_m), '| Divorced:', df_em_m['is_divorced'].sum())

formula_div = '~ log_income + education + age + age_squared + is_jawa + is_sunda'

m2_f, cols2_f = run_weighted_model(df_em_f, 'is_divorced', formula_div, 'Ever Divorced â€” Women (Base)')
m2_m, cols2_m = run_weighted_model(df_em_m, 'is_divorced', formula_div, 'Ever Divorced â€” Men (Base)')
tbl_m2 = format_results_table(m2_f, cols2_f, m2_m, cols2_m, 'TABLE 3: EVER DIVORCED â€” BASE MODEL')

---
## Considering Urban/Rural Control
We now add `urban` (1 = Urban, 0 = Rural) to assess whether the urbanization effect confounds
the income and ethnicity coefficients. Urban individuals tend to earn more and marry later,
so omitting this variable could bias the income and Jawa/Sunda coefficients upward in magnitude.

## 12. Model 1b â€” Ever Married (Extended: With Urban/Rural)

In [ ]:
formula_urban = '~ log_income + education + age + age_squared + is_jawa + is_sunda + urban'

m1b_f, cols1b_f = run_weighted_model(df_female, 'ever_married', formula_urban, 'Ever Married â€” Women (+ Urban)')
m1b_m, cols1b_m = run_weighted_model(df_male,   'ever_married', formula_urban, 'Ever Married â€” Men (+ Urban)')
tbl_m1b = format_results_table(m1b_f, cols1b_f, m1b_m, cols1b_m, 'TABLE 2b: EVER MARRIED â€” WITH URBAN/RURAL')

## 13. Model 2b â€” Ever Divorced (Extended: With Urban/Rural)

In [ ]:
m2b_f, cols2b_f = run_weighted_model(df_em_f, 'is_divorced', formula_urban, 'Ever Divorced â€” Women (+ Urban)')
m2b_m, cols2b_m = run_weighted_model(df_em_m, 'is_divorced', formula_urban, 'Ever Divorced â€” Men (+ Urban)')
tbl_m2b = format_results_table(m2b_f, cols2b_f, m2b_m, cols2b_m, 'TABLE 3b: EVER DIVORCED â€” WITH URBAN/RURAL')

## 14. Coefficient Comparison: Base vs. Urban/Rural Extended

In [ ]:
def compare_coefs(m_base, m_ext, cols_base, cols_ext, sex, outcome):
    shared = [c for c in cols_base if c in cols_ext]
    rows = []
    for col in shared:
        i_base = cols_base.index(col)
        i_ext  = cols_ext.index(col)
        rows.append({
            'Variable'  : col,
            'Base Coef.': round(m_base.params[i_base], 3),
            'Ext Coef.' : round(m_ext.params[i_ext],  3),
            'Change'    : round(m_ext.params[i_ext] - m_base.params[i_base], 3),
        })
    tbl = pd.DataFrame(rows).set_index('Variable')
    print(f'\n=== {outcome} â€” {sex}: Base vs. + Urban ===')
    print(tbl.to_string())

# Ever Married
compare_coefs(m1_f,  m1b_f,  cols1_f,  cols1b_f,  'Women', 'Ever Married')
compare_coefs(m1_m,  m1b_m,  cols1_m,  cols1b_m,  'Men',   'Ever Married')

# Ever Divorced
compare_coefs(m2_f,  m2b_f,  cols2_f,  cols2b_f,  'Women', 'Ever Divorced')
compare_coefs(m2_m,  m2b_m,  cols2_m,  cols2b_m,  'Men',   'Ever Divorced')

## 15. Predicted Probability Plots (Extended Model)

In [ ]:
def plot_predicted_prob(m_f, m_m, df_f, df_m, pred_cols, outcome_label, filename):
    income_range = np.linspace(df_clean['log_income'].quantile(0.05),
                               df_clean['log_income'].quantile(0.95), 100)
    other_cols = [c for c in pred_cols if c not in ('Intercept','log_income')]
    fig, ax = plt.subplots(figsize=(9, 5))
    for model, df_sub, color, label in [
        (m_f, df_f, '#e74c3c', 'Women'),
        (m_m, df_m, '#2980b9', 'Men'),
    ]:
        means = df_sub[other_cols].mean()
        pred_df = pd.DataFrame({'Intercept': 1.0, 'log_income': income_range,
                                **{c: means[c] for c in other_cols}})
        prob = model.predict(pred_df[pred_cols].values)
        ax.plot(income_range, prob, color=color, label=label, linewidth=2)
    ax.set_xlabel('Log Monthly Income', fontsize=12)
    ax.set_ylabel('Predicted Probability', fontsize=12)
    ax.set_title(outcome_label, fontsize=13)
    ax.legend()
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()

plot_predicted_prob(m1b_f, m1b_m, df_female, df_male, cols1b_f,
                   'Predicted Probability: Ever Married by Log Income (+ Urban)',
                   'ever_married_urban.png')
plot_predicted_prob(m2b_f, m2b_m, df_em_f, df_em_m, cols2b_f,
                   'Predicted Probability: Ever Divorced by Log Income (+ Urban)',
                   'ever_divorced_urban.png')

## End

In [ ]:
# Side-by-side comparison: Base SE vs Cluster-Robust SE
print('=== COMPARISON: Standard SE vs Cluster-Robust SE ===')
print('(Coefficients are identical; only SEs differ)\n')

for sex, m_base, m_clust, cols in [
    ('Women â€” Ever Married', m1b_f, m1b_f, cols1b_f),
    ('Men   â€” Ever Married', m1b_m, m1b_m, cols1b_m),
    ('Women â€” Ever Divorced', m2b_f, m2b_f, cols2b_f),
    ('Men   â€” Ever Divorced', m2b_m, m2b_m, cols2b_m),
]:
    rows = []
    for i, col in enumerate(cols):
        se_base    = m_base.bse[i]
        se_cluster = m_clust.bse[i]
        p_base     = m_base.pvalues[i]
        p_cluster  = m_clust.pvalues[i]
        rows.append({
            'Variable':    col,
            'Coef.':       round(m_base.params[i], 4),
            'SE (base)':   round(se_base, 4),
            'SE (cluster)':round(se_cluster, 4),
            'SE change':   f'{(se_cluster/se_base - 1)*100:+.1f}%',
            'p (base)':    round(p_base, 4),
            'p (cluster)': round(p_cluster, 4),
        })
    df_cmp = pd.DataFrame(rows).set_index('Variable')
    print(f'\n--- {sex} ---')
    print(df_cmp.to_string())

In [ ]:
# Merge hhid14 back in (needed for clustering)
cov_hhid = load_stata('adult_a/b3a_cov.dta', ['pidlink', 'hhid14'])
df_female_c = df_female.merge(cov_hhid, on='pidlink', how='left')
df_male_c   = df_male.merge(cov_hhid,   on='pidlink', how='left')

df_em_fc = df_em_f.merge(cov_hhid, on='pidlink', how='left')
df_em_mc = df_em_m.merge(cov_hhid, on='pidlink', how='left')

print('hhid14 non-null â€” Female:', df_female_c['hhid14'].notna().sum(),
      '| Male:', df_male_c['hhid14'].notna().sum())


def run_cluster_model(df_sub, outcome, formula_rhs, label):
    pred_cols = [c.strip() for c in formula_rhs.replace('~', '').split('+')]
    df_m = df_sub.dropna(subset=[outcome] + pred_cols + ['pwt14xa', 'hhid14']).copy()
    n = len(df_m)
    w = df_m['pwt14xa'].values
    w_norm = w / w.sum() * n
    groups = df_m['hhid14'].values

    y, X = patsy.dmatrices(f'{outcome} {formula_rhs}', data=df_m, return_type='dataframe')
    model = sm.Logit(y.values.ravel(), X, freq_weights=w_norm).fit(
        disp=False,
        cov_type='cluster',
        cov_kwds={'groups': groups}
    )
    pr2 = 1 - model.llf / model.llnull
    res = pd.DataFrame({
        'Coef.':    model.params,
        'Cluster SE': model.bse,
        'z':        model.tvalues,
        'P>|z|':    model.pvalues,
        'OR':       np.exp(model.params),
    }, index=X.columns)
    print(f'\n--- {label} (N={n:,}, Pseudo RÂ²={pr2:.4f}) ---')
    print(res.round(4).to_string())
    return model, X.columns.tolist()


formula_urban = '~ log_income + education + age + age_squared + is_jawa + is_sunda + urban'

print('\n=== EVER MARRIED â€” CLUSTER-ROBUST (SURVEY-DESIGN-CORRECTED) ===')
mc1_f, ccols1_f = run_cluster_model(df_female_c, 'ever_married', formula_urban, 'Women: Ever Married')
mc1_m, ccols1_m = run_cluster_model(df_male_c,   'ever_married', formula_urban, 'Men: Ever Married')

print('\n=== EVER DIVORCED â€” CLUSTER-ROBUST (SURVEY-DESIGN-CORRECTED) ===')
mc2_f, ccols2_f = run_cluster_model(df_em_fc, 'is_divorced', formula_urban, 'Women: Ever Divorced')
mc2_m, ccols2_m = run_cluster_model(df_em_mc, 'is_divorced', formula_urban, 'Men: Ever Divorced')

In [ ]:
print('\n=== EVER DIVORCED â€” CLUSTER-ROBUST (SURVEY-DESIGN-CORRECTED) ===')
mc2_f, ccols2_f = run_cluster_model(df_em_fc, 'is_divorced', formula_urban, 'Women: Ever Divorced')
mc2_m, ccols2_m = run_cluster_model(df_em_mc, 'is_divorced', formula_urban, 'Men: Ever Divorced')

---
## 16. Survey-Design-Corrected Models (Cluster-Robust Standard Errors)

Hopcroft (2021) used **PROC SURVEYLOGISTIC** in SAS, which accounts for the clustered survey design.
We replicate this in Python by adding `cov_type='cluster'` with `hhid14` as the cluster variable.

**Difference from base models:**
- Coefficients are identical (same weighted estimates)
- Standard errors are **larger** (corrected for within-household correlation)
- This is the Python equivalent of SAS `PROC SURVEYLOGISTIC` / R `svyglm`

In [ ]:
from scipy import stats

def diff_table(m_f, cols_f, m_m, cols_m, title):
    shared = [c for c in cols_f if c in cols_m]
    rows = []
    for col in shared:
        i_f = cols_f.index(col)
        i_m = cols_m.index(col)
        b_f, se_f = m_f.params[i_f], m_f.bse[i_f]
        b_m, se_m = m_m.params[i_m], m_m.bse[i_m]
        diff  = b_f - b_m
        se_d  = np.sqrt(se_f**2 + se_m**2)
        z     = diff / se_d
        p     = 2 * (1 - stats.norm.cdf(abs(z)))
        stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        rows.append({"Variable": col, "Women Coef": round(b_f,3), "Women SE": round(se_f,3),
                     "Men Coef": round(b_m,3), "Men SE": round(se_m,3),
                     "Diff": round(diff,3), "SE Diff": round(se_d,3),
                     "z": round(z,3), "p": round(p,4), "Sig": stars})
    tbl = pd.DataFrame(rows).set_index("Variable")
    sep = "=" * 70
    print(f"\n{sep}")
    print(f"  {title}")
    print(sep)
    print(f"{'Variable':<15} {'Women':>8} {'(SE)':>7} {'Men':>8} {'(SE)':>7} {'Diff':>8} {'(SE)':>7} {'Sig':>5}")
    print("-" * 70)
    for _, r in tbl.reset_index().iterrows():
        print(f"{r['Variable']:<15} {r['Women Coef']:>8.3f} ({r['Women SE']:>5.3f}) "
              f"{r['Men Coef']:>8.3f} ({r['Men SE']:>5.3f}) "
              f"{r['Diff']:>8.3f} ({r['SE Diff']:>5.3f}) {r['Sig']:>5}")
    def diff_table(m_f, cols_f, m_m, cols_m, title):
        shared = [c for c in cols_f if c in cols_m]
        rows = []
        for col in shared:
            i_f = cols_f.index(col)
            i_m = cols_m.index(col)
            b_f, se_f = m_f.params[i_f], m_f.bse[i_f]
            b_m, se_m = m_m.params[i_m], m_m.bse[i_m]
            diff  = b_f - b_m
            se_d  = np.sqrt(se_f**2 + se_m**2)
            z     = diff / se_d
            p     = 2 * (1 - stats.norm.cdf(abs(z)))
            stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
            rows.append({"Variable": col, "Women Coef": round(b_f,3), "Women SE": round(se_f,3),
                         "Men Coef": round(b_m,3), "Men SE": round(se_m,3),
                         "Diff": round(diff,3), "SE Diff": round(se_d,3),
                         "z": round(z,3), "p": round(p,4), "Sig": stars})
        tbl = pd.DataFrame(rows).set_index("Variable")
        sep = "=" * 70
        print(f"\n{sep}")
        print(f"  {title}")
        print(sep)
        print(f"{'Variable':<15} {'Women':>8} {'(SE)':>7} {'Men':>8} {'(SE)':>7} {'Diff':>8} {'(SE)':>7} {'Sig':>5}")
        print("-" * 70)
        for _, r in tbl.reset_index().iterrows():
            print(f"{r['Variable']:<15} {r['Women Coef']:>8.3f} ({r['Women SE']:>5.3f}) {r['Men Coef']:>8.3f} ({r['Men SE']:>5.3f}) {r['Diff']:>8.3f} ({r['SE Diff']:>5.3f}) {r['Sig']:>5}")
        print(f"\nN Women={int(m_f.nobs):,}  N Men={int(m_m.nobs):,}")
        print("* p<0.05  ** p<0.01  *** p<0.001")
        return tbl
    print("* p<0.05  ** p<0.01  *** p<0.001")
    return tbl

In [ ]:
# First, run the base model (without urban) to get mc1b_* variables
formula_base = '~ log_income + education + age + age_squared + is_jawa + is_sunda'

print('\n=== EVER MARRIED - CLUSTER-ROBUST BASE (WITHOUT URBAN) ===')
mc1b_f, ccols1b_f = run_cluster_model(df_female_c, 'ever_married', formula_base, 'Women: Ever Married (Base)')
mc1b_m, ccols1b_m = run_cluster_model(df_male_c,   'ever_married', formula_base, 'Men: Ever Married (Base)')

# Now compare base vs urban models
diff_table(mc1b_f, ccols1b_f, mc1b_m, ccols1b_m,
           "EVER MARRIED — BASE MODEL (Sex Difference, Cluster-Robust SE)")

diff_table(mc1_f, ccols1_f, mc1_m, ccols1_m,
           "EVER MARRIED — WITH URBAN (Sex Difference, Cluster-Robust SE)")

In [ ]:
print('\n=== EVER DIVORCED - CLUSTER-ROBUST BASE (WITHOUT URBAN) ===')
mc2b_f, ccols2b_f = run_cluster_model(df_em_fc, 'is_divorced', formula_base, 'Women: Ever Divorced (Base)')
mc2b_m, ccols2b_m = run_cluster_model(df_em_mc, 'is_divorced', formula_base, 'Men: Ever Divorced (Base)')

diff_table(mc2b_f, ccols2b_f, mc2b_m, ccols2b_m,
           "EVER DIVORCED — BASE MODEL (Sex Difference, Cluster-Robust SE)")

In [ ]:
diff_table(mc2_f, ccols2_f, mc2_m, ccols2_m,
           "EVER DIVORCED — WITH URBAN (Sex Difference, Cluster-Robust SE)")

In [ ]:
formula_base = '~ log_income + education + age + age_squared + is_jawa + is_sunda'

print('\n=== EVER MARRIED - CLUSTER-ROBUST BASE (WITHOUT URBAN) ===')
mc1b_f, ccols1b_f = run_cluster_model(df_female_c, 'ever_married', formula_base, 'Women: Ever Married (Base)')
mc1b_m, ccols1b_m = run_cluster_model(df_male_c,   'ever_married', formula_base, 'Men: Ever Married (Base)')

formula_base = '~ log_income + education + age + age_squared + is_jawa + is_sunda'

print('\n=== EVER MARRIED - CLUSTER-ROBUST BASE (WITHOUT URBAN) ===')
mc1b_f, ccols1b_f = run_cluster_model(df_female_c, 'ever_married', formula_base, 'Women: Ever Married (Base)')
mc1b_m, ccols1b_m = run_cluster_model(df_male_c,   'ever_married', formula_base, 'Men: Ever Married (Base)')

print('\n=== EVER DIVORCED - CLUSTER-ROBUST BASE (WITHOUT URBAN) ===')
mc2b_f, ccols2b_f = run_cluster_model(df_em_fc, 'is_divorced', formula_base, 'Women: Ever Divorced (Base)')
mc2b_m, ccols2b_m = run_cluster_model(df_em_mc, 'is_divorced', formula_base, 'Men: Ever Divorced (Base)')
mc2b_f, ccols2b_f = run_cluster_model(df_em_fc, 'is_divorced', formula_base, 'Women: Ever Divorced (Base)')
mc2b_m, ccols2b_m = run_cluster_model(df_em_mc, 'is_divorced', formula_base, 'Men: Ever Divorced (Base)')

In [ ]:
diff_table(mc1_f, ccols1_f, mc1_m, ccols1_m, "EVER MARRIED — WITH URBAN...")